# 05 · The preregistered 15-item intervention grid and its geometry (Appendices `app:grid`, `app:intervention`, `app:models`, deviations)

> **Provenance.** Built with the `repro-notebook` workflow (one executed notebook per experiment,
> every reported number checked against the produced one). Target paper: *Conversational Relevance in the
> Conjunction Fallacy Is Represented in Language Models* (`docs/paper/main_A.tex` in the authors' working
> repository; the released copy of the manuscript is the submission PDF). Companion documents:
> `REPRODUCTION_MANIFEST.md` (paper number → source CSV → producing script) and `REPRODUCTION_REPORT.md`
> (aggregated verdicts). Status legend: ✅ reproduced to the reported precision · ❌ does not match ·
> ⚠ presence check only · ⛔ cannot be recomputed from the released files.

## Source and code map

| id | what | data read here | original producing script |
|---|---|---|---|
| E1 | ablation / injection cell tests, guards | `results/rq3_intervention_final_20260827/analysis/rq3b_abl_tests.csv`, `rq3b_add_tests.csv` | `analysis/rq3/analyze_rq3_intervention.py` ← `run_rq3_intervention_gpu.py`, `run_rq3_fluency_guard_gpu.py` |
| E1 | manipulation check of the grid (norm shifts under ablation) | `rq3f_R_specificity.csv`, `rq3f_abl_norming_tests.csv` | `analysis/rq3/analyze_rq3_dis.py` ← `run_rq3_dis_gpu.py` |
| E2, E3, E5 | LOO cosine, k-sweep capture, decoding vs capture | `rq3x_subspace_k_k14_verify.csv`, `rq3x_subspace_k_k2.csv`, `rq3a_context_decoder_robustness.csv` | `analysis/rq3/verify_discussion_numbers.py`, `analyze_rq3_subspace_k.py` (need the activations to re-run; outputs shipped) |
| E11 | directions and doses of the grid | `rq3b_abl_tests.csv`, `rq3b_add_tests.csv` | as E1 |
| E8, E12 | compute and environment | `results/*/manifests/*.json` | run manifests |

Everything is **loaded**; the cell-level aggregates of E1 are recomputed from the stored tests.

In [1]:
import sys, warnings
sys.path.insert(0, str(__import__('pathlib').Path.cwd() / 'notebooks'))
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from repro_utils import *
pd.set_option('display.width', 200, 'display.max_columns', 40, 'display.max_rows', 60,
              'display.float_format', lambda v: f'{v:,.4f}')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25,
                     'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 9})
print('repository root:', ROOT.name)
C = Checks('05_preregistered_grid')

repository root: release


## E1 · The preregistered grid returned a null

Reported (Appendix `app:intervention`): ablation and injection each supported the hypothesis in **1 of 10** adjudicated cells;
layer-matched ablation *raises* the DoD (**+0.009**) and injection lowers it in all four arms (**−0.019 to −0.032**); the only
directions that move as predicted are **two controls**; the preregistered ablation cell clears the guards in **0 of 40** cells.

Adjudication here: Holm-corrected $p<.05$, predicted sign, guard passed. Layer-matched ablation is `ablate_scope == selected`
at $\lambda=1$; injection is the `selected_safe` strength per cell. The four arms are `R_h3`, `R_h3_denoised`, `R_h3_k2`, `R_h3_orth_PC`.

In [2]:
abl = read(GRID / 'rq3b_abl_tests.csv'); add = read(GRID / 'rq3b_add_tests.csv')
ARMS = ['R_h3', 'R_h3_denoised', 'R_h3_k2', 'R_h3_orth_PC']
sel = abl[(abl.ablate_scope == 'selected') & (abl.dose == 1.0)]
def supported(d, sign):
    return int(((d.holm_task_p < .05) & (np.sign(d['mean']) == sign) & d.guard_pass.astype(bool)).sum())
a = sel[sel.direction == 'R_h3']; i = add[(add.direction == 'R_h3') & (add.selected_safe == True)]
C.frac('E1', 'Appendix app:intervention', 'ablation of v_R supports the hypothesis (adjudicated cells)', '1/10', supported(a, -1), len(a), rel(GRID / 'rq3b_abl_tests.csv'))
C.frac('E1', 'Appendix app:intervention', 'injection of v_R supports the hypothesis (adjudicated cells)', '1/10', supported(i, 1), len(i), rel(GRID / 'rq3b_add_tests.csv'))
C.num('E1', 'Appendix app:intervention', 'layer-matched ablation, mean ΔDoD over 10 cells (R_h3, λ=1)', '+0.009', a['mean'].mean(),
      note='recomputed mean over the 10 model × task cells; the paper\'s +0.009 may average a different cell set')
inj = pd.Series({d: add[(add.direction == d) & (add.selected_safe == True)]['mean'].mean() for d in ARMS}); display(inj.rename('mean ΔDoD, injection at selected safe α'))
C.rng('E1', 'Appendix app:intervention', 'injection lowers the DoD in all four arms (range of arm means)', '-0.032–-0.019', inj.min(), inj.max(), decimals=3,
      note='all four arm means are negative; the printed range differs from the recomputed one')
pre = abl[(abl.ablate_scope == 'all') & (abl.dose == 1.0) & abl.direction.isin(ARMS)]
C.frac('E1', 'Appendix app:intervention', 'preregistered ablation cells clearing both guards (4 arms × 10 cells)', '0/40', pre.guard_pass.astype(bool).sum(), len(pre))
g = sel.groupby('direction').apply(lambda d: pd.Series({'mean': d['mean'].mean(), 'supported': supported(d, -1)}), include_groups=False)
display(g.sort_values('mean'))
C.frac('E1', 'Appendix app:intervention', 'directions moving as predicted under layer-matched ablation are controls', '2/2', int((g.supported > 0).sum() - g.loc[ARMS].supported.gt(0).sum()), int((g.supported > 0).sum()),
       note='directions with ≥1 supported cell: ' + ', '.join(g.index[g.supported > 0]))

R_h3            -0.0319
R_h3_denoised   -0.0379
R_h3_k2         -0.0250
R_h3_orth_PC    -0.0297
Name: mean ΔDoD, injection at selected safe α, dtype: float64

,mean,supported
direction,,
C,-0.0074,1.0000
R_h2,-0.0048,0.0000
random_k2_01,-0.0015,0.0000
random_k2_00,-0.0013,1.0000
random_k2_07,-0.0009,0.0000
P,-0.0007,0.0000
random_k2_03,-0.0006,0.0000
shuffle,-0.0004,0.0000
random,0.0008,0.0000


### The manipulation check of the grid does not pass

Reported: re-asking the norming questions with the hook in place, *no cell* has all four intervals containing zero (**0 of 20**);
ablating $v_{R,h_3}$, the $h_2$ control, the bare control, $v_P$, $v_C$ or their span all shift $C$ by **+2.5 to +3.8**.
The released `rq3f_R_specificity.csv` holds the ablation-induced norm shifts per (model, task, direction, measure, hypothesis).

In [3]:
rs = read(GRID / 'rq3f_R_specificity.csv'); display(rs.head(3))
print('directions:', sorted(rs.direction.unique()), '· stages:', sorted(rs.stage.unique()), '· metrics:', sorted(rs.metric.unique()))
# four intervals per (model, task, direction, stage): metric × hypothesis; a cell passes if all four contain zero
zero = rs.groupby(['model_id', 'task', 'direction', 'stage']).ci_contains_zero.all()
per = zero.groupby(['direction', 'stage']).agg(['sum', 'size']); display(per)
tot = zero.xs('R_h3', level='direction')
C.frac('E1', 'Appendix app:intervention', 'grid manipulation check for v_R: cells (10 ablation + 10 injection) with all four norm intervals containing zero', '0/20', tot.sum(), len(tot),
       rel(GRID / 'rq3f_R_specificity.csv'))
cshift = rs[(rs.stage == 'ABL') & (rs.metric == 'C') & (rs.hypothesis == 'h3')].groupby('direction').mean_change.mean(); display(cshift.rename('mean ΔC(h3) under ablation'))
nt = read(GRID / 'rq3f_abl_norming_tests.csv')
allC = pd.concat([cshift[['R_h3', 'R_h2', 'R_base']], nt.groupby('direction')['mean'].mean()])
display(allC.rename('mean C shift under ablation, six directions'))
C.rng('E1', 'Appendix app:intervention', 'C shift under ablation across v_R, h2, bare, v_P, v_C, span (range of direction means)', '2.5–3.8', allC.min(), allC.max(), decimals=1,
      note='rq3f_abl_norming_tests.csv stores the P/C/PC ablation shifts on a different scale (means 0.8–1.2); the printed range is not recovered from the release')


,model_id,task,direction,stage,alpha,metric,hypothesis,mean_change,ci_lo,ci_hi,ci_contains_zero,p_flip_two
0,Qwen/Qwen2.5-Math-1.5B,2,R_base,ADD,1.0000,rho,h2,-0.8347,-1.1019,-0.5676,False,0.0001
1,Qwen/Qwen2.5-Math-1.5B,2,R_base,ADD,1.0000,rho,h3,-0.7237,-0.9466,-0.5007,False,0.0001
2,Qwen/Qwen2.5-Math-1.5B,2,R_base,ADD,1.0000,C,h2,0.6280,0.4707,0.7853,False,0.0001


directions: ['R_base', 'R_h2', 'R_h3', 'R_h3_denoised', 'R_h3_k2', 'R_h3_orth_PC', 'R_pooled'] · stages: ['ABL', 'ADD'] · metrics: ['C', 'rho']


sum  size
direction     stage           
R_base        ABL      0    10
              ADD      0     8
R_h2          ABL      0    10
              ADD      1     8
R_h3          ABL      0    10
              ADD      0    10
R_h3_denoised ADD      0    10
R_h3_k2       ADD      0    10
R_h3_orth_PC  ADD      0    10
R_pooled      ABL      0    10
              ADD      0    10

direction
R_base     3.8747
R_h2       3.3428
R_h3       2.3676
R_pooled   1.0712
Name: mean ΔC(h3) under ablation, dtype: float64

direction
R_h3     2.3676
R_h2     3.3428
R_base   3.8747
C        1.1891
P        0.7961
PC       1.0692
Name: mean C shift under ablation, six directions, dtype: float64

## E2 · Fifteen items read a direction but do not steer one: the LOO cosine is +0.271 (74.3°), opposite side in 14 of 150 folds

Source: `rq3x_subspace_k_k14_verify.csv`, produced by `verify_discussion_numbers.py` from the canonical activations
(the activations are not shipped, the output is). Ten cells: five models × two wordings that passed the decoder gate in this run.

In [4]:
k14 = read(GRID / 'rq3x_subspace_k_k14_verify.csv'); display(k14[['model', 'task', 'layer', 'n', 'loo_cos_mean', 'loo_cos_negative_folds', 'heldout_capture_k1', 'heldout_capture_k14', 'push_cos_k1_k14']])
m = k14.loo_cos_mean.mean()
C.num('E2', 'Appendix app:intervention', 'mean LOO cosine between the training axis and the held-out contrast', '+0.271', m, rel(GRID / 'rq3x_subspace_k_k14_verify.csv'))
C.num('E2', 'Appendix app:intervention', 'the same cosine as an angle (degrees)', '74.3', float(np.degrees(np.arccos(m))))
C.frac('E2', 'Appendix app:intervention', 'folds where the axis falls on the opposite side', '14/150', k14.loo_cos_negative_folds.sum(), k14.n.sum())
C.num('E2', 'Appendix app:intervention', 'the component an edit can move is a fifth of the contrast (capture at k=1)', '0.178', k14.heldout_capture_k1.mean())

,model,task,layer,n,loo_cos_mean,loo_cos_negative_folds,heldout_capture_k1,heldout_capture_k14,push_cos_k1_k14
0,Qwen2.5-Math-1.5B,2,14,15,0.1306,4,0.0730,0.2194,0.4870
1,Qwen2.5-Math-1.5B,3,14,15,0.1419,3,0.0783,0.2268,0.5132
2,Qwen3-4B-Instruct-2507,2,22,15,0.2654,1,0.1705,0.2719,0.7532
3,Qwen3-4B-Instruct-2507,3,20,15,0.2941,1,0.2115,0.3207,0.7679
4,Qwen3-4B,2,22,15,0.2259,1,0.1177,0.2113,0.6752
5,Qwen3-4B,3,20,15,0.2519,2,0.1537,0.2485,0.7135
6,gemma-3-4b-it,2,34,15,0.2766,1,0.1914,0.3095,0.6971
7,gemma-3-4b-it,3,34,15,0.3083,1,0.2423,0.3460,0.7338
8,Phi-4-mini-instruct,2,10,15,0.3991,0,0.2657,0.3601,0.8483
9,Phi-4-mini-instruct,3,10,15,0.4146,0,0.2787,0.3713,0.8575


## E3 · Higher-dimensional subspaces do not recover the contrast (appendix table)

Reported: capture 0.178 / 0.212 / 0.242 / 0.289 at $k=1,2,5,14$; random 0.00039 / 0.00097 / 0.00218 / 0.00592; ratios
451× / 219× / 111× / 49×; push direction within cosine 0.705 of the one-dimensional one. Averaged over the ten cells.

In [5]:
rows = []
for kk in [1, 2, 3, 5, 14]:
    c = k14[f'heldout_capture_k{kk}'].mean(); r = k14[f'heldout_capture_random_k{kk}'].mean()
    rows.append(dict(k=kk, capture=c, random=r, ratio=c / r))
tab = pd.DataFrame(rows).set_index('k'); display(tab.style.format({'capture': '{:.3f}', 'random': '{:.5f}', 'ratio': '{:.0f}×'}))
for kk, cap, rnd, ratio in [(1, '0.178', '0.00039', '451'), (2, '0.212', '0.00097', '219'), (5, '0.242', '0.00218', '111'), (14, '0.289', '0.00592', '49')]:
    C.num('E3', 'Appendix table (k-sweep)', f'capture at k={kk}', cap, tab.loc[kk, 'capture'], rel(GRID / 'rq3x_subspace_k_k14_verify.csv'),
          note='the k=2 row in the paper (0.212, 219×) does not match the stored k=2 capture' if kk == 2 else '')
    C.num('E3', 'Appendix table (k-sweep)', f'random capture at k={kk}', rnd, tab.loc[kk, 'random'],
          note='the random column depends on the RNG stream; no stored file reproduces the printed random values')
    C.num('E3', 'Appendix table (k-sweep)', f'ratio at k={kk}', ratio, tab.loc[kk, 'ratio'], note='follows the random column')
C.num('E3', 'Appendix app:intervention', 'push direction cosine between k=1 and k=14', '0.705', k14.push_cos_k1_k14.mean())
k2 = read(GRID / 'rq3x_subspace_k_k2.csv'); print('k=2 capture in the separate k2 file:', round(k2.heldout_capture_k2.mean(), 4), '(same construction)')

,capture,random,ratio
k,,,
1,0.178,0.00036,491×
2,0.204,0.00075,271×
3,0.226,0.00137,165×
5,0.242,0.00213,113×
14,0.289,0.00572,50×


k=2 capture in the separate k2 file: 0.2042 (same construction)


## E5 · Decoding and capture are dissociated: Spearman ρ = +0.20, p = 0.58

Reported: the cell with the lowest capture (0.073) decodes at 0.711 and the one with the highest (0.279) at 0.671.
This recomputation settles a discrepancy in the working notes, which quoted +0.31, p = .33 from an older run.

In [6]:
gate = read(GRID / 'rq3a_context_decoder_robustness.csv'); gate = gate[gate.family == 'conjunction_h3']; gate['model_s'] = gate.model.str.split('/').str[-1]
mm = k14.merge(gate[['model_s', 'task', 'auc_nested_loio']], left_on=['model', 'task'], right_on=['model_s', 'task'])
rho, p = stats.spearmanr(mm.heldout_capture_k1, mm.auc_nested_loio)
display(mm[['model', 'task', 'heldout_capture_k1', 'auc_nested_loio']].sort_values('heldout_capture_k1'))
C.num('E5', 'Appendix app:intervention', 'Spearman ρ between capture and decoding AUC', '+0.20', rho, rel(GRID / 'rq3x_subspace_k_k14_verify.csv'))
C.num('E5', 'Appendix app:intervention', 'its p value', '0.58', p)
lo = mm.loc[mm.heldout_capture_k1.idxmin()]; hi = mm.loc[mm.heldout_capture_k1.idxmax()]
C.num('E5', 'Appendix app:intervention', 'lowest-capture cell: capture', '0.073', lo.heldout_capture_k1); C.num('E5', 'Appendix app:intervention', 'lowest-capture cell: AUC', '0.711', lo.auc_nested_loio)
C.num('E5', 'Appendix app:intervention', 'highest-capture cell: capture', '0.279', hi.heldout_capture_k1); C.num('E5', 'Appendix app:intervention', 'highest-capture cell: AUC', '0.671', hi.auc_nested_loio)

,model,task,heldout_capture_k1,auc_nested_loio
0,Qwen2.5-Math-1.5B,2,0.0730,0.7111
1,Qwen2.5-Math-1.5B,3,0.0783,0.5867
4,Qwen3-4B,2,0.1177,0.7067
5,Qwen3-4B,3,0.1537,0.7200
2,Qwen3-4B-Instruct-2507,2,0.1705,0.6933
6,gemma-3-4b-it,2,0.1914,0.8400
3,Qwen3-4B-Instruct-2507,3,0.2115,0.6578
7,gemma-3-4b-it,3,0.2423,0.8178
8,Phi-4-mini-instruct,2,0.2657,0.7244
9,Phi-4-mini-instruct,3,0.2787,0.6711


## E4 · Variance decomposition (item × model 20.8 % / 24.4 %, arm main effect 0.04–0.11 %) and E7 · retired decoder labelling (AUC 0.559)

Neither quantity has a stored output or a producing script in the released files.

In [7]:
C.blocked('E4', 'Appendix app:intervention', 'variance decomposition of the ablation/injection effect (item × model share)', '20.8% / 24.4%; arm 0.04–0.11%',
          'no stored output or script located; would require the per-item edit tables rq3b_*_item_effects and a decomposition not in the release')
C.blocked('E7', 'Deviation 1', 'retired one-versus-three rank AUC', '0.559 (6/12)', 'the retired gate file is not part of the canonical run; the value exists only in an internal notebook output')
C.blocked('E1', 'Appendix app:grid', 'the manipulation check denominator "20 cells" and the "+2.5 to +3.8" range across six directions', '0/20; +2.5 to +3.8',
          'partially reproduced above from rq3f_R_specificity.csv; the exact cell set of the printed statement is not recoverable from the release')

## E11 · The grid: twenty ablation directions, seventeen injection directions, λ ∈ {0.25, 0.5, 1}, α ∈ {0.5, 1, 2, 4}

In [8]:
C.frac('E11', 'Appendix app:grid', 'ablation directions', '20/20', abl.direction.nunique(), 20, rel(GRID / 'rq3b_abl_tests.csv'))
C.frac('E11', 'Appendix app:grid', 'injection directions', '17/17', add.direction.nunique(), 17, rel(GRID / 'rq3b_add_tests.csv'))
C.presence('E11', 'Appendix app:grid', 'ablation doses λ', '{0.25, 0.5, 1.0}', str(sorted(abl.dose.unique())))
C.presence('E11', 'Appendix app:grid', 'injection strengths α', '{0.5, 1, 2, 4}', str(sorted(add.alpha.unique())))
print('ablation directions:', sorted(abl.direction.unique()))

ablation directions: ['C', 'P', 'PC', 'R_base', 'R_h2', 'R_h3', 'R_h3_denoised', 'R_h3_k2', 'R_h3_orth_PC', 'R_pooled', 'random', 'random_k2_00', 'random_k2_01', 'random_k2_02', 'random_k2_03', 'random_k2_04', 'random_k2_05', 'random_k2_06', 'random_k2_07', 'shuffle']


## E8, E12 · Models, assets and compute (Appendix `app:models`)

Reported: torch 2.9.1 (CUDA 12.8), transformers 4.57.1; one NVIDIA TITAN RTX (24 GB); the preregistered grid took 56 GPU-hours
summed over five models at peak allocation 18.8 GB; pinned model revisions. Read from the run manifests (server paths scrubbed).

In [9]:
import json, glob
man = [json.load(open(p)) for p in sorted(glob.glob(str(GRID.parent / 'manifests' / '*.json')))]
def walk(o, key):
    if isinstance(o, dict):
        for k, v in o.items():
            if k == key: yield v
            yield from walk(v, key)
    elif isinstance(o, list):
        for v in o: yield from walk(v, key)
versions = {(v, t) for m in man for v in walk(m, 'torch_version') for t in walk(m, 'transformers_version')}
print('torch / transformers versions in manifests:', versions)
C.presence('E12', 'Appendix app:models', 'pinned library versions', 'torch 2.9.1 / transformers 4.57.1', '; '.join(sorted(f'{a} / {b}' for a, b in versions)))
peak_mib = max([x for m in man for x in walk(m, 'peak_nvidia_mib') if isinstance(x, (int, float))] or [0])
print(f'peak allocation over the shipped manifests: {peak_mib:.0f} MiB = {peak_mib/1024:.2f} GiB')
C.blocked('E8', 'Appendix app:models', 'GPU-hours of the preregistered grid', '56 h over five models', 'the shipped manifests record memory and versions but no wall-clock time')
C.num('E8', 'Appendix app:models', 'peak GPU allocation (MiB / 1000, as printed)', '18.8', peak_mib / 1000, note=f'{peak_mib:.0f} MiB = {peak_mib/1024:.2f} GiB; the paper divides MiB by 1000')
revs = {m.get('model'): r for m in man for r in walk(m, 'model_revision')}
C.presence('E12', 'Appendix app:models', 'pinned model revisions', '1cfa9a7, cdbee75, 4a83ca6, 093f9f3, cfbefac, 0e3b1e2', '; '.join(f'{k}: {str(v)[:7]}' for k, v in revs.items() if k))

torch / transformers versions in manifests: {('2.9.1+cu128', '4.57.1')}
peak allocation over the shipped manifests: 18808 MiB = 18.37 GiB


## Reproduction table for this notebook

In [10]:
df = C.show(); C.save('05')

05_preregistered_grid: ✅ 21 · ❌ 13 · ⛔ 4 · ⚠ 4


id,section,analysis,reported,produced,status,note
E1,Appendix app:intervention,ablation of v_R supports the hypothesis (adjudicated cells),1/10,1/10,✅,
E1,Appendix app:intervention,injection of v_R supports the hypothesis (adjudicated cells),1/10,1/10,✅,
E1,Appendix app:intervention,"layer-matched ablation, mean ΔDoD over 10 cells (R_h3, λ=1)",+0.009,+0.0129,❌,recomputed mean over the 10 model × task cells; the paper's +0.009 may average a different cell set
E1,Appendix app:intervention,injection lowers the DoD in all four arms (range of arm means),-0.032–-0.019,-0.038–-0.025,❌,all four arm means are negative; the printed range differs from the recomputed one
E1,Appendix app:intervention,preregistered ablation cells clearing both guards (4 arms × 10 cells),0/40,0/40,✅,
E1,Appendix app:intervention,directions moving as predicted under layer-matched ablation are controls,2/2,2/3,❌,"directions with ≥1 supported cell: C, R_h3, random_k2_00"
E1,Appendix app:intervention,grid manipulation check for v_R: cells (10 ablation + 10 injection) with all four norm intervals containing zero,0/20,0/20,✅,
E1,Appendix app:intervention,"C shift under ablation across v_R, h2, bare, v_P, v_C, span (range of direction means)",2.5–3.8,0.8–3.9,❌,rq3f_abl_norming_tests.csv stores the P/C/PC ablation shifts on a different scale (means 0.8–1.2); the printed range is not recovered from the release
E2,Appendix app:intervention,mean LOO cosine between the training axis and the held-out contrast,+0.271,+0.2708,✅,
E2,Appendix app:intervention,the same cosine as an angle (degrees),74.3,+74.29,✅,


saved notebooks/_checks_05.csv
